# Basic Input/Output

**Topic:** `1.7` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*I/O in Python is an object you can replace. This topic covers print and the format specification, the stdout/stderr contract, and the redirection technique that makes console programs testable without a terminal.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

Every robotics service begins as a console program, and a deployment script usually captures its output. Getting the two streams right is what makes `service > state.json` produce something a parser can read, and passing the stream in is what makes the program testable at all.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain that print writes to sys.stdout and input reads from sys.stdin.
2. Redirect a stream to capture output in a test.
3. Format aligned columns with f-string format specifications.
4. Apply the stdout/stderr contract to a reporting function.
5. Detect end of input correctly in a read loop.
6. Build a report without quadratic string concatenation.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.6 Operators and Expressions

## 4. Mental Model

**Mental model: three pipes, each with a name you can swap.** stdin carries data in, stdout carries results out, stderr carries complaints out. `print` and `input` are simply the default functions attached to those pipes. Change the name on the pipe and every function using it follows - which is why redirecting output in a test changes the behaviour of the code under test without changing the code.

| You write | Actually happens |
| --- | --- |
| `print(x)` | str(x) is written to `sys.stdout` |
| `input(p)` | p is written to stdout, a line read from `sys.stdin` |
| `print(x, file=sys.stderr)` | x is written to the error pipe instead |
| `sys.stdout = buffer` | every later print goes to the buffer |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Stream | A file-like object carrying text between a program and its environment. |
| stdin | The standard input stream, conventionally file descriptor 0. |
| stdout | The standard output stream, conventionally file descriptor 1. |
| stderr | The standard error stream, conventionally file descriptor 2. |
| Buffer | An in-memory stream that can stand in for a terminal or a file. |
| redirect_stdout | A context manager that swaps sys.stdout for the duration of a block. |
| Format specification | The `:>8.1f` style spec that controls width and precision. |
| flush | Forcing buffered output to be written immediately. |
| End of input | An empty read from a stream, which is not the same as an empty line. |
| CLI contract | The convention that stdout holds results and stderr holds diagnostics. |

## 6. Conceptual Explanation

Input and output in Python are objects, not language constructs. `print` is a function that writes to `sys.stdout`; `input` is a function that reads a line from `sys.stdin`. Because those are ordinary module attributes, you can replace them. That single fact is what makes an interactive program testable without a human at the keyboard, and it is the most useful thing in this topic.

In [ ]:
import sys

print('first')
print('second')

# print takes any number of objects and one separator
print('rover-01', 48.0, True, sep=' | ')
print('no newline here', end='')
print(' - appended')

print('stdout is:', sys.stdout.write.__self__ is sys.stdout)

Formatted output is where most console programs earn their keep. The f-string format specification controls alignment, width, precision and type, and using it is far more readable than `str.format` or `%` for anything with a column layout.

In [ ]:
rows = [('rover-01', 48.0), ('rover-02', 12.5), ('rover-03', 99.9)]

print(f"{'ROBOT':<10} {'BATTERY':>8}")
print('-' * 18)
for name, pct in rows:
    print(f'{name:<10} {pct:>7.1f}%')

> **Why input() is dangerous in a notebook**
>
> input() blocks waiting for a human, and in a notebook it will hang the kernel with no way to recover except interrupting it. Throughout this course, input is demonstrated by redirecting sys.stdin from an in-memory stream, which runs deterministically and immediately.

## 7. Formal Theory

Every running Python process has three standard streams, each a text-mode file object: `sys.stdin` for input, `sys.stdout` for normal output, and `sys.stderr` for diagnostics. The first two are text streams with a defined encoding; `sys.stderr` is unbuffered and line-oriented by convention. A command-line tool is expected to put machine-readable results on stdout and human-facing diagnostics on stderr, so the two can be redirected independently.

```text
process  ->  fd 0 (stdin)   fd 1 (stdout)   fd 2 (stderr)
```

Because `sys.stdout` is an object, an assignment rebinds the name that `print` looks up at call time. This is how `contextlib.redirect_stdout` captures output in a test, and how a program can write to a string, a file or a socket by changing one attribute.

In [ ]:
import io
import sys

buffer = io.StringIO()
original = sys.stdout
sys.stdout = buffer
try:
    print('captured by the buffer')
finally:
    sys.stdout = original

print('buffered text:', repr(buffer.getvalue()))
print('restored  ->', 'print still writes here')

| Stream | Purpose | Consequence of writing there |
| --- | --- | --- |
| `sys.stdin` | Data the program consumes | Blocks if nothing is piped in |
| `sys.stdout` | Results a caller may parse | Breaks `prog > file.json` |
| `sys.stderr` | Diagnostics for a human | Bypasses the parsed output |

## 8. Syntax

`print` accepts any number of objects, one `sep` between them and an `end` after the last. Both are strings, and both are what make column output readable without a single concatenation.

In [ ]:
def print_header(title: str, width: int = 24) -> None:
    print(title.center(width, '='))


print_header('ROBO-X')
print('id', 'battery', 'status', sep='\t')
print('rover-01', 48.0, 'ACTIVE', sep='\t')
print()                                   # an empty line
print('done', end='\n', flush=True)      # explicit end and flush

The anti-pattern:

```text
print('Robot: ' + name + ' battery: ' + str(pct) + '%')

print(f'{name}: {pct}%')     # one expression, no conversion calls
```

> **input() always returns a string**
>
> Whatever the user types arrives as text, so a number must be converted before arithmetic. Forgetting that produces the classic TypeError: can only concatenate str to str.

## 9. Basic Examples

**Example 1 - reading from a redirected stdin instead of a human.**

In [ ]:
import io
import sys


def read_answer(prompt: str, answers) -> str:
    """Stand-in for input() that reads from a fixed list of answers."""
    print(prompt, end=' ')
    return next(answers)


answers = iter(['rover-01', '48.5'])
name = read_answer('robot id?', answers)
battery = read_answer('battery pct?', answers)

print(f'parsed -> {name} at {float(battery)}%')

**Example 2 - the string methods every parser needs.**

In [ ]:
raw = '  Rover-01  '
print('strip     :', repr(raw.strip()))
print('lower     :', raw.strip().lower())
print('split     :', 'front,rear,left'.split(','))
print('partition :', 'key=value'.partition('='))
print('replace   :', 'rover-01'.replace('-', '_'))
print('startswith:', raw.strip().startswith('Ro'))
print('zfill     :', '7'.zfill(3))

## 10. Intermediate Examples

Real stdout has the same contract, which is what makes a console tool composable: print a header to the standard output, and diagnostics to the error stream, so the two can be separated downstream.

In [ ]:
import sys


def report(robot_id: str, battery_pct: float) -> None:
    print(f'{robot_id}:{battery_pct:.1f}')          # machine-readable
    print(f'warning: {robot_id} battery is low', file=sys.stderr)


report('rover-01', 12.5)
print('the two streams can be redirected independently')

## 11. Advanced Examples

Capturing output turns a printing function into a testable one. This is the single most useful technique in this topic, because it means a console program can be verified in an automated suite without a human present.

In [ ]:
import io
import sys
from contextlib import redirect_stdout


def menu() -> None:
    print('1. arm')
    print('2. disarm')


buffer = io.StringIO()
with redirect_stdout(buffer):
    menu()

captured = buffer.getvalue()
print('captured lines:', captured.strip().splitlines())
print('contains arm  :', 'arm' in captured)

## 12. Code Walkthrough

Below is an interactive dispatch prompt written so it can be tested. It reads from an injected stream and returns a decision rather than acting on one.

In [ ]:
"""dispatch.py - ask for a command and turn it into a decision."""
from __future__ import annotations

COMMANDS = {
    '1': 'ARM',
    '2': 'DISARM',
    'q': 'QUIT',
}


def read_choice(stream) -> str:
    """Read one line from a file-like object, tolerating EOF."""
    line = stream.readline()
    if not line:
        return 'q'
    return line.strip().lower()


def run(stream) -> list[str]:
    """Process commands until quit, returning the decisions taken."""
    print('ROBO-X dispatch: 1=arm 2=disarm q=quit')
    history: list[str] = []
    while True:
        print('> ', end='', flush=True)
        choice = read_choice(stream)
        action = COMMANDS.get(choice)
        if action is None:
            print(f'unknown command: {choice!r}')
            continue
        if action == 'QUIT':
            break
        history.append(action)
        print(f'  -> {action}')
    return history

Two design choices make this testable. The stream is a parameter rather than a global read of `input`, so a test supplies a `StringIO`. And `readline` returning an empty string is treated as end of input, which means a closed pipe ends the loop instead of spinning forever.

## 13. Line-by-Line Explanation

1. COMMANDS is a module-level lookup table, so adding a command is one entry rather than another branch in the loop.
2. read_choice takes the stream as a parameter, which is the whole reason the program can be tested without a keyboard.
3. readline returning an empty string is the end-of-input signal and maps to 'q', so a closed pipe terminates the loop instead of spinning.
4. strip().lower() normalises the input so 'Q' and ' q ' behave like 'q'.
5. The while loop continues on an unknown command rather than quitting, which is what a human expects from a prompt.
6. Actions are appended to history instead of being executed, so the function is pure with respect to the robot and its output can be asserted directly.

## 14. Common Mistakes

**Mistake 1 - forgetting that input returns text.**

```text
value = input('battery: ')   # always a str
value * 2                    # TypeError

value = float(input('battery: '))
```

**Mistake 2 - printing diagnostics to stdout.**

```text
print('rover-01:48.0')       # result, parseable downstream
print('warning: low battery')  # oops, also on stdout

print('warning: low battery', file=sys.stderr)
```

**Mistake 3 - using input() in a notebook.**

```text
name = input('id: ')   # blocks the kernel with no visible prompt
```

**Mistake 4 - a loop that never terminates on end of input.**

```text
while input() != 'q':   # raises EOFError at the end, or spins
    process()

line = stream.readline()
if not line:            # explicit end-of-input check
    break
```

## 15. Debugging Techniques

The first question about any console problem is which stream the text went to. Redirect each one separately and the answer becomes obvious.

In [ ]:
import sys

print('this goes to stdout')
print('this goes to stderr', file=sys.stderr)

# python -c '...' > out.txt        captures stdout only
# python -c '...' 2> err.txt      captures stderr only
# python -c '...' > all.txt 2>&1  captures both

When a formatted column looks wrong, print the raw values alongside the formatted ones. A width specifier silently truncates or pads, so comparing `repr(value)` with the rendered cell usually reveals the mismatch.

In [ ]:
value = 1234.5678
print('raw        :', repr(value))
print('8.1f       :', f'{value:8.1f}')
print('8.3f       :', f'{value:8.3f}')
print('left, 10.4f:', f'{value:<10.4f}|')
print('explicit  :', f'[{value:8.1f}]')

> **flush when it matters**
>
> Output is buffered when it is not a terminal, so a long-running program may appear to hang. Pass flush=True to print, or run python with -u, when the user needs to see progress as it happens.

## 16. Best Practices

- Put machine-readable results on stdout and diagnostics on stderr.
- Use f-string format specs for columns rather than manual padding.
- Convert input to the type you need immediately, at the point of reading.
- Read from a stream object passed in, not from a global `input` call.
- Treat an empty read as end of input and exit the loop.
- Capture output with redirect_stdout so console code can be tested.
- Use flush=True for progress output in long-running services.
- Keep a prompt on the same line with print(..., end='', flush=True).

## 17. Performance Considerations

Console I/O is almost always the bottleneck, and it is a property of the terminal rather than of Python. A single `print` may trigger a write syscall; a loop of ten thousand prints will visibly lag, while a loop of a million in-memory operations will not. If a program must emit a large volume of text, build one string and write it once rather than calling print repeatedly.

The second cost is string concatenation. Building a report with `result += line` inside a loop is quadratic, because each concatenation copies everything written so far. A list with `append` followed by a single `join` is linear. This is one of the few cases where the idiomatic Python version is not merely prettier but asymptotically faster.

> **Batch your writes**
>
> For a large report, accumulate lines in a list and print the joined result once, or write to a file handle with a large buffer.

## 18. Pythonic Approaches

Joining a report: accumulating versus collecting then joining.

In [ ]:
rows = [('rover-01', 48.0), ('rover-02', 12.5)]

# before: repeated concatenation copies the whole string each time
report = ''
for name, pct in rows:
    report += f'{name} {pct:.1f}\n'

# after: build a list, join once
lines = [f'{name} {pct:.1f}' for name, pct in rows]
report = '\n'.join(lines)

print(report)

## 19. Robotics Connection

Robot services are console programs. They print a banner at start-up, report readings while running, and send warnings to stderr so a supervisor can log them separately from the data stream. Getting the two streams right is what makes `service > telemetry.json` work.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()

print(f"{state['name']}:{state['battery_pct']:.1f}")          # data
if state['battery_pct'] < 20:
    print('warning: low battery', file=sys.stderr)  # diagnostic

## 20. Engineering Example

A reporting function that writes machine-readable output to one stream and human-readable warnings to the other.

In [ ]:
import sys


def report_fleet(robots, stream=sys.stdout, errors=sys.stderr) -> None:
    """Print one parseable line per robot and warnings to the error stream."""
    for robot in robots:
        name = robot.get('name', '<unnamed>')
        pct = robot.get('battery_pct')
        if pct is None:
            print(f'warning: {name} has no battery reading', file=errors)
            continue
        print(f'{name}:{pct:.1f}', file=stream)
        if pct < 20:
            print(f'warning: {name} battery {pct:.1f}%', file=errors)

## 21. Guided Practice

1. Print a two-column table with the robot name left-aligned and the battery right-aligned to eight characters.
2. Capture the output of a small function with redirect_stdout and assert that it contains what you expect.
3. Read three lines from a StringIO stream and echo each one with a prefix.
4. Write a function that prints its data to a stream parameter and its warnings to sys.stderr, then call it twice.

In [ ]:
import io
from contextlib import redirect_stdout

def banner() -> None:
    print('ROBO-X ready')


buffer = io.StringIO()
with redirect_stdout(buffer):
    banner()
assert 'ROBO-X ready' in buffer.getvalue()
print('captured:', repr(buffer.getvalue()))

stream = io.StringIO('rover-01\nrover-02\n')
while (line := stream.readline()):
    print('echo:', line.strip())

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** Where does `print('x')` send its text?
- **Q1.** What does `contextlib.redirect_stdout` do?
- **Q1.** Why does a long-running service appear to hang when its output is redirected to a file?
- **Q1.** A robot supervisor parses the output of a status command. Where should human warnings go?

## 23. Summary

Input and output in Python are objects, not language features. `print` writes to `sys.stdout`, `input` reads from `sys.stdin`, and both are ordinary module attributes you can replace. That is the central idea of this topic, and it has an immediate practical consequence: an interactive program can be tested without a human, by supplying an in-memory stream.

The second half of the topic is formatting and the stream contract. The f-string format specification handles alignment, width and precision well enough to replace manual padding entirely. And the convention that stdout carries machine-readable results while stderr carries human diagnostics is what makes a console tool composable - it is the reason `roboctl status > state.json` produces a file you can parse.

The practical habit to take away is to pass the stream in rather than reaching for a global. A function that takes a stream is testable, scriptable and reusable; a function that calls `input` directly is none of those. And when you write a loop that reads until a sentinel, treat an empty read as end of input - otherwise a closed pipe turns your prompt into an infinite loop.

## 24. Key Takeaways

- `print` writes to `sys.stdout`; `input` reads from `sys.stdin`.
- Both are replaceable objects, which is what makes console code testable.
- Redirecting stdout is how a test asserts on printed output.
- Format specs control alignment, width and precision without manual padding.
- `input()` always returns a string, so convert before arithmetic.
- Put results on stdout and diagnostics on stderr.
- Pass the stream in as a parameter rather than calling input directly.
- An empty read means end of input; always check it in a read loop.

## 25. Further Exploration

- [input and output](https://docs.python.org/3/tutorial/inputoutput.html)
- [sys - standard streams and configuration](https://docs.python.org/3/library/sys.html)
- [io - core tools for working with streams](https://docs.python.org/3/library/io.html)
- [contextlib.redirect_stdout](https://docs.python.org/3/library/contextlib.html#contextlib.redirect_stdout)
- [Format specification mini-language](https://docs.python.org/3/library/string.html#format-specification-mini-language)